In [5]:
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import nfl_data_py as nfl

schedule_data = pd.read_csv(r'2025_schedule.csv')

In [6]:
columns = [
    'season','game_type','week','gameday','weekday','gametime','away_team','home_team','home_coach','away_coach',
    'away_qb_id','home_qb_id','away_qb_name','home_qb_name','surface','away_rest','home_rest'
]

schedule_df_final = schedule_data[columns]

In [7]:
# Select only the relevant columns
columns = ['passer_player_id','passer_player_name', 'posteam', 'defteam', 'season', 'week', 'home_team', 'away_team', 'play_type', 'air_yards', 
           'yards_after_catch', 'epa', 'complete_pass', 'incomplete_pass', 'interception', 'qb_hit', 'sack', 'pass_touchdown',
           'passing_yards', 'cpoe', 'roof', 'surface','drive','game_seconds_remaining', 'game_id']

# Loading in the NFL pbp data
data = nfl.import_pbp_data(range(2010,2026), downcast=True)
data = data[columns]

# nfl-data-py still loads other columns, so we again need to set our data equal to only the columns we want
data = data[columns]

# Drop all rows that are not a pass
data = data[data['play_type'] == 'pass']

# Drop the play type column
passer_data = data.drop(columns=['play_type'])

# Group the data together by passer, week, season and aggregate
passer_df = passer_data.groupby(['passer_player_name', 'game_id', 'passer_player_id', 'week', 'season'], as_index=False).agg(
    {'posteam' : 'first',
     'defteam' : 'first',
     'home_team' : 'first',
     'away_team' : 'first',
     'air_yards' : 'sum',
     'yards_after_catch' : 'sum',
     'epa' : 'sum',
     'complete_pass' : 'sum',
     'incomplete_pass' : 'sum',
     'interception' : 'sum',
     'qb_hit' : 'sum',
     'sack' : 'sum',
     'pass_touchdown' : 'sum',
     'passing_yards' : 'sum',
     'cpoe' : 'mean',
     'roof' : 'first',
     'surface' : 'first'
     }
)

# Create a new column that is completion percentage
passer_df['completion_percentage'] = passer_df['complete_pass'] / (passer_df['complete_pass'] + passer_df['incomplete_pass'])

# Create a new column that is the number of pass attempts
passer_df['pass_attempts'] = passer_df['complete_pass'] + passer_df['incomplete_pass']

# Drop the complete_pass and incomplete_pass columns
passer_df = passer_df.drop(columns=['complete_pass', 'incomplete_pass'])

# Create a new column that equals 1 if the passer is the home team and 0 if the passer is the away team
passer_df['home_flag'] = passer_df['home_team'] == passer_df['posteam']

# Drop the home_team and away_team columns
passer_df = passer_df.drop(columns=['home_team', 'away_team'])

# Reorder the columns
#passer_df = passer_df[['passer_player_name','passer_player_id', 'posteam', 'defteam', 'season', 'week', 'passing_yards', 'home_flag', 'completion_percentage', 'pass_attempts',
#                       'air_yards',  'yards_after_catch', 'epa', 'interception', 'qb_hit', 'sack', 'pass_touchdown', 
#                        'cpoe', 'roof', 'surface']]

drives = data[['game_id', 'drive', 'game_seconds_remaining', 'posteam']].copy()

# Sort the data by game and drive to ensure correct ordering
drives = drives.sort_values(by=['game_id', 'drive', 'game_seconds_remaining'], ascending=[True, True, False])

# Drop duplicates to get the start and end of each drive
drive_starts = drives.groupby(['game_id', 'drive']).first().reset_index()
drive_ends = drives.groupby(['game_id', 'drive']).last().reset_index()

# Merge the start and end times to calculate the duration
drive_durations = pd.merge(drive_starts, drive_ends, on=['game_id', 'drive', 'posteam'], suffixes=('_start', '_end'))

# Calculate the duration of each drive in seconds
drive_durations['drive_duration_seconds'] = drive_durations['game_seconds_remaining_start'] - drive_durations['game_seconds_remaining_end']

# Group by game and team to get total TOP
game_top = drive_durations.groupby(['game_id', 'posteam']).agg(
    total_top_seconds=('drive_duration_seconds', 'sum')
).reset_index()

passer_df_merged = passer_df.merge(game_top, on=['game_id', 'posteam'], how='inner')
passer_df_merged.drop('game_id', axis=1, inplace=True)

ewma_cols = ['completion_percentage', 'pass_attempts', 'air_yards', 'yards_after_catch', 'epa', 'interception', 
             'qb_hit', 'sack', 'pass_touchdown', 'passing_yards', 'cpoe','total_top_seconds']

passer_df = passer_df_merged.sort_values(by=['passer_player_name', 'season', 'week']).reset_index(drop=True)

for col in ewma_cols:
    new_col_name = f'{col}_ewma'
    
    # Group by player, apply EWM (which is calculated cumulatively), and then shift
    passer_df[new_col_name] = passer_df.groupby('passer_player_name')[col]\
        .transform(lambda x: x.ewm(min_periods=1, span=10).mean().shift(1))


# 3. Drop the non-ewma columns
passer_df = passer_df.drop(columns=ewma_cols)

# Select only the relevant columns
defense_columns = ['defteam', 'season', 'week', 'home_team', 'away_team', 'play_type', 'air_yards',
                   'yards_after_catch', 'epa', 'complete_pass', 'incomplete_pass', 'interception', 'qb_hit', 'sack', 'pass_touchdown',
                   'passing_yards', 'cpoe', 'roof', 'surface']


# nfl-data-py still loads other columns, so we again need to set our data equal to only the columns we want
defense_data = data[defense_columns]

# Drop the play type column
defense_data = defense_data.drop(columns=['play_type'])

# Group the data together by passer, week, season and aggregate
defense_df = defense_data.groupby(['defteam', 'week', 'season'], as_index=False).agg(
    {'home_team': 'first',
     'away_team': 'first',
     'air_yards': 'sum',
     'yards_after_catch': 'sum',
     'epa': 'sum',
     'complete_pass': 'sum',
     'incomplete_pass': 'sum',
     'interception': 'sum',
     'qb_hit': 'sum',
     'sack': 'sum',
     'pass_touchdown': 'sum',
     'passing_yards': 'sum',
     'cpoe': 'mean',
     'roof': 'first',
     'surface': 'first'
     }
)

# Create a new column that is completion percentage
defense_df['completion_percentage'] = defense_df['complete_pass'] / (defense_df['complete_pass'] + defense_df['incomplete_pass'])

# Create a new column that is the number of pass attempts
defense_df['pass_attempts'] = defense_df['complete_pass'] + defense_df['incomplete_pass']

# Drop the complete_pass and incomplete_pass columns
defense_df = defense_df.drop(columns=['complete_pass', 'incomplete_pass'])

# Create a new column that equals 1 if the defense is the home team and 0 if the defense is the away team
defense_df['home_flag'] = defense_df['home_team'] == defense_df['defteam']

# Drop the home_team and away_team columns
defense_df = defense_df.drop(columns=['home_team', 'away_team'])

# Reorder the columns
defense_df = defense_df[['defteam', 'season', 'week', 'home_flag', 'passing_yards', 'completion_percentage', 'pass_attempts',
                       'air_yards',  'yards_after_catch', 'epa', 'interception', 'qb_hit', 'sack', 'pass_touchdown', 
                       'cpoe', 'roof', 'surface']]

# Calculate the exponentially weighted moving average for each feature
defense_df['completion_percentage_ewma'] = defense_df.groupby('defteam')['completion_percentage']\
    .transform(lambda x: x.ewm(min_periods=1, span=10).mean())

defense_df['pass_attempts_ewma'] = defense_df.groupby('defteam')['pass_attempts']\
    .transform(lambda x: x.ewm(min_periods=1, span=10).mean())

defense_df['air_yards_ewma'] = defense_df.groupby('defteam')['air_yards']\
    .transform(lambda x: x.ewm(min_periods=1, span=10).mean())

defense_df['yards_after_catch_ewma'] = defense_df.groupby('defteam')['yards_after_catch']\
    .transform(lambda x: x.ewm(min_periods=1, span=10).mean())

defense_df['epa_ewma'] = defense_df.groupby('defteam')['epa']\
    .transform(lambda x: x.ewm(min_periods=1, span=10).mean())

defense_df['interception_ewma'] = defense_df.groupby('defteam')['interception']\
    .transform(lambda x: x.ewm(min_periods=1, span=10).mean())

defense_df['qb_hit_ewma'] = defense_df.groupby('defteam')['qb_hit']\
    .transform(lambda x: x.ewm(min_periods=1, span=10).mean())

defense_df['sack_ewma'] = defense_df.groupby('defteam')['sack']\
    .transform(lambda x: x.ewm(min_periods=1, span=10).mean())

defense_df['pass_touchdown_ewma'] = defense_df.groupby('defteam')['pass_touchdown']\
    .transform(lambda x: x.ewm(min_periods=1, span=10).mean())

defense_df['passing_yards_ewma'] = defense_df.groupby('defteam')['passing_yards']\
    .transform(lambda x: x.ewm(min_periods=1, span=10).mean())

defense_df['cpoe_ewma'] = defense_df.groupby('defteam')['cpoe']\
    .transform(lambda x: x.ewm(min_periods=1, span=10).mean())

# Drop the non-ewma columns
defense_df = defense_df.drop(columns=['passing_yards','completion_percentage', 'pass_attempts', 'air_yards', 'yards_after_catch', 'epa', 
                                    'interception', 'qb_hit', 'sack', 'pass_touchdown', 'cpoe'])

# Merge the defense and passer dataframes together
df = passer_df.merge(defense_df, how='inner', on=['defteam', 'season', 'week', 'roof', 'surface'], suffixes=('_passer', '_defense'))
df = df[df['pass_attempts_ewma_passer'] > 5]

final_df = df.dropna()

2010 done.
2011 done.
2012 done.
2013 done.
2014 done.
2015 done.
2016 done.
2017 done.
2018 done.
2019 done.
2020 done.
2021 done.
2022 done.
2023 done.
2024 done.
2025 done.
Downcasting floats.


MemoryError: Unable to allocate 1.11 GiB for an array with shape (205, 727093) and data type float64

In [ ]:
import joblib
import numpy as np

week_num = 3

# Step 1: Load the saved model from the .joblib file.
try:
    model = joblib.load('lgbm_passing_model.joblib')
    modelq10 = joblib.load('lgbm_passing_TD_q10_model.joblib')
    modelq50 = joblib.load('lgbm_passing_TD_q50_model.joblib')
    modelq90 = joblib.load('lgbm_passing_TD_q90_model.joblib')
    print("Model loaded successfully.")
except FileNotFoundError:
    print("Error: The .joblib file was not found. Please check the file path and name.")
    #exit()


# Step 3: Prepare the data for prediction.
# We need to filter for Week 1 games and merge them with the most recent player and defensive stats.

# Filter for Week 1 games from the schedule data.
week_1_games = schedule_df_final[schedule_df_final['week'] == week_num].copy()

# A helper function to find the most recent stats for a player or team
def get_most_recent_stats(df, name, name_col, team_col, team):
    """
    Finds the most recent stats for a given player or defensive team.
    """
    # Filter for the specific player/team
    filtered_df = df[(df[name_col] == name) & (df[team_col] == team)].copy()
    
    # Sort by season and week to find the most recent game
    if not filtered_df.empty:
        most_recent_game = filtered_df.sort_values(by=['season', 'week'], ascending=False).iloc[0]
        return most_recent_game
    return None

# Step 3a: Extract the full feature list from the training data.
# This list is crucial to ensure the prediction data has the same columns.
features_from_training = pd.read_csv(r'model_trained_on_data.csv').columns.tolist()
features_from_training.remove('Unnamed: 0')
if 'pass_touchdown' in features_from_training:
    features_from_training.remove('pass_touchdown')

# Step 3b: Create a list of dictionaries for each QB in Week 1 games, populated with stats.
prediction_rows = []
for index, row in week_1_games.iterrows():
    # --- Process Home QB
    home_qb_name = row['home_qb_id']
    home_team = row['home_team']
    home_qb_stats = get_most_recent_stats(final_df, home_qb_name, 'passer_player_id', 'posteam', home_team)
    
    if home_qb_stats is not None:
        home_qb_row = home_qb_stats.to_dict()
        # Add schedule-specific info that the model might need
        home_qb_row['home_flag_passer'] = True # Home QB is playing at home
        home_qb_row['home_team'] = row['home_team']
        home_qb_row['away_team'] = row['away_team']
        home_qb_row['season'] = row['season']
        home_qb_row['week'] = row['week']
        
        # Add defensive stats for the opposing team (the 'away' team in this game)
        away_def_stats = get_most_recent_stats(final_df, row['away_team'], 'defteam', 'defteam', row['away_team'])
        if away_def_stats is not None:
            for col in away_def_stats.keys():
                if 'defense' in col:
                    home_qb_row[col] = away_def_stats[col]

        prediction_rows.append(home_qb_row)

    # --- Process Away QB
    away_qb_name = row['away_qb_id']
    away_team = row['away_team']
    away_qb_stats = get_most_recent_stats(final_df, away_qb_name, 'passer_player_id', 'posteam', away_team)
    
    if away_qb_stats is not None:
        away_qb_row = away_qb_stats.to_dict()
        away_qb_row['home_flag_passer'] = False # Away QB is not playing at home
        away_qb_row['home_team'] = row['home_team']
        away_qb_row['away_team'] = row['away_team']
        away_qb_row['season'] = row['season']
        away_qb_row['week'] = row['week']

        # Add defensive stats for the opposing team (the 'home' team in this game)
        home_def_stats = get_most_recent_stats(final_df, row['home_team'], 'defteam', 'defteam', row['home_team'])
        if home_def_stats is not None:
            for col in home_def_stats.keys():
                if 'defense' in col:
                    away_qb_row[col] = home_def_stats[col]

        prediction_rows.append(away_qb_row)

# Create a DataFrame from the combined data
if not prediction_rows:
    print("Warning: No Week 1 games with available stats found. Cannot make predictions.")
    #exit()

week_1_qbs_df = pd.DataFrame(prediction_rows)
print("\nPrepared Week 1 data with combined player and defensive stats.")

# Store the original player and team names before one-hot encoding
prediction_info = week_1_qbs_df[['passer_player_name', 'home_team', 'away_team']].copy()

# Step 3c: One-hot encode the categorical features to match the training data.
categorical_features = ['passer_player_name', 'posteam', 'defteam', 'home_team', 'away_team']
week_1_dummies = pd.get_dummies(week_1_qbs_df.drop(['passer_player_id'], axis=1), columns=categorical_features, drop_first=True, dtype=int)

# Step 3d: Align the columns to ensure they match the training data perfectly.
# Any column in the training data not present in the prediction data will be filled with 0.
X_predict = week_1_dummies.reindex(columns=features_from_training, fill_value=0)

# Step 4: Use the loaded model to predict the passing yards.
predictions = model.predict(X_predict.drop(['season','week'], axis=1))
predictions10 = modelq10.predict(X_predict.drop(['season','week'], axis=1))
predictions50 = modelq50.predict(X_predict.drop(['season','week'], axis=1))
predictions90 = modelq90.predict(X_predict.drop(['season','week'], axis=1))

# Step 5: Add the predictions to your DataFrame.
X_predict['predicted_passing_TD'] = predictions
X_predict['predicted_passing_TD_Q10'] = predictions10
X_predict['predicted_passing_TD_Q50'] = predictions50
X_predict['predicted_passing_TD_Q90'] = predictions90
X_predict['interval_range_pass_TD'] = X_predict['predicted_passing_TD_Q90'] - X_predict['predicted_passing_TD_Q10']

#individual_predictions = np.array([tree.predict(X_predict) for tree in model.estimators_])
#prediction_std = np.std(individual_predictions, axis=0)

# Add standard deviation to the DataFrame
#X_predict['prediction_std'] = prediction_std

# Calculate 95% confidence intervals (approx. 1.96 standard deviations)
#confidence_interval = 1.96 * prediction_std
#X_predict['ci_lower_bound'] = X_predict['predicted_passing_yards'] - confidence_interval
#X_predict['ci_upper_bound'] = X_predict['predicted_passing_yards'] + confidence_interval


# Step 6: Join the original player/team names back to the predictions DataFrame.
X_predict = pd.concat([prediction_info, X_predict], axis=1)

# Step 6: Join the original player/team names back to the predictions DataFrame.
X_predict = pd.concat([prediction_info, X_predict], axis=1)
# Step 6: Display the results.
print(f"\nWeek {week_num} Predictions with Passer Names:")
#print(X_predict[['passer_player_name', 'home_team', 'away_team', 'predicted_passing_yards', 'ci_lower_bound', 'ci_upper_bound']])
print(X_predict[['passer_player_name', 'home_team', 'away_team', 'predicted_passing_TD','predicted_passing_TD_Q10','predicted_passing_TD_Q50','predicted_passing_TD_Q90','interval_range_pass_TD']])

from datetime import datetime
X_predict[['passer_player_name', 'week','season', 'home_team', 'away_team', 'predicted_passing_TD','predicted_passing_TD_Q10','predicted_passing_TD_Q50','predicted_passing_TD_Q90','interval_range_pass_TD']].to_csv(fr"Preds\{datetime.today().strftime('%Y-%m-%d')}_predictions_passing_TDs_{week_num}.csv", index=False)

Model loaded successfully.

Prepared Week 1 data with combined player and defensive stats.


ValueError: Number of features of the model must match the input. Model n_features_ is 328 and input n_features is 329

In [ ]:
X_predict = X_predict.loc[:, ~X_predict.columns.duplicated()]

Index(['passer_player_name', 'week', 'season', 'home_team', 'away_team',
       'predicted_passing_yards', 'predicted_passing_yards_Q10',
       'predicted_passing_yards_Q50', 'predicted_passing_yards_Q90',
       'interval_range'],
      dtype='object')
Index(['index', 'passer_player_name', 'passing_yds_var', 'season'], dtype='object')


In [36]:
hist = pd.read_csv(fr"historical_evaluation_{datetime.today().strftime('%Y-%m-%d')}.csv")
hist = hist.groupby('passer_player_name', as_index=False).agg({'passing_yds_var':'mean', 'season':'count'}).reset_index()
hist.rename({'season':'num_games'}, inplace=True, axis=1)
hist = X_predict[['passer_player_name', 'week','season', 'home_team', 'away_team', 'predicted_passing_yards','predicted_passing_yards_Q10','predicted_passing_yards_Q50','predicted_passing_yards_Q90','interval_range']].merge(hist, on=['passer_player_name'])
hist.to_csv(r'hist_eval.csv')

In [37]:
hist

,passer_player_name,week,season,home_team,away_team,predicted_passing_yards,predicted_passing_yards_Q10,predicted_passing_yards_Q50,predicted_passing_yards_Q90,interval_range,index,passing_yds_var,num_games
0,J.Allen,3,2025,BUF,MIA,257.659558,169.120638,250.972113,342.755356,173.634718,11,-46.082994,2
1,T.Tagovailoa,3,2025,BUF,MIA,237.758764,151.774165,239.091721,318.915782,167.141617,29,33.485758,2
2,B.Young,3,2025,CAR,ATL,214.111034,108.050627,235.121695,311.652525,203.601898,4,-47.975928,2
3,M.Penix,3,2025,CAR,ATL,230.805251,113.584785,220.298584,303.205332,189.620547,23,-0.700491,2
4,J.Flacco,3,2025,CLE,GB,248.323174,147.640807,250.611729,330.507796,182.866988,15,-1.142398,2
5,J.Love,3,2025,CLE,GB,231.668383,106.417797,231.711029,310.612026,204.194229,19,-12.362001,2
6,T.Lawrence,3,2025,JAX,HOU,221.730208,86.611779,215.741839,298.296524,211.684745,28,-28.949315,2
7,C.Stroud,3,2025,JAX,HOU,210.740260,87.232830,204.360470,298.407140,211.174310,5,-0.102446,2
8,D.Maye,3,2025,NE,PIT,193.407544,67.763441,201.977387,310.414473,242.651031,8,-98.918648,2
9,A.Rodgers,3,2025,NE,PIT,253.685486,148.793044,260.915365,349.341888,200.548843,0,40.677745,1
